# Agente de estudo — Transformers e risco de crédito PF
## Parte I — Compreender e executar a LLM local

**Objetivo:** construir progressivamente um assistente para estudar Transformers e propor experimentos em risco de crédito Pessoa Física.

Optou-se por um único notebook, com código direto, configurações explícitas e seções cronológicas. O estilo didático aproveita o EuroSAT e os Projetos 6–9 de NLP; não se reproduzem seus datasets nem resultados.

**Entrega desta primeira parte:** carregar uma única LLM, inspecionar uma conversa tokenizada, gerar uma resposta e compreender a entrada do LangChain. As Partes II a VI implementam documentos, banco vetorial, RAG, ajuste e interface. A Parte I permanece uma introdução sem RAG.

| Ordem | Conteúdo | Estado |
|---|---|---|
| I | Ambiente, Qwen, tokens, geração e prompts | Implementado neste notebook |
| II | Curadoria, PDF, chunks e embeddings | Próxima etapa |
| III | Chroma, HNSW, cosseno e avaliação de MMR | Planejado |
| IV | RAG com fontes e chat baseado no Projeto 8 | C?digo implementado; gera??o completa pendente |
| V | Dataset PF, QLoRA e comparação antes/depois | Planejado |

Não se utilizará uma segunda LLM para comparar modelos ou avaliar respostas. Um encoder de embeddings será necessário posteriormente: ele representa trechos para busca e não é outra LLM geradora.

## 1. Problema e escolha da LLM

Uma pergunta como “Como usar atenção sobre 12 meses de comportamento financeiro?” exige conceitos, código e desenho de validação. Uma resposta plausível pode estar errada; o futuro RAG fornecerá fontes e o ajuste será avaliado separadamente.

Optou-se por **Qwen/Qwen2.5-3B-Instruct**, da família/tamanho já utilizada via Ollama no curso. A escolha considera seguimento de instruções, português, continuidade com Qwen e tentativa de execução local em 8 GB. É uma escolha de engenharia, não uma comprovação de que seja universalmente melhor ou superior no domínio PF. Não haverá benchmark 1,5B versus 3B.

O carregamento será via Transformers, com pesos em 4 bits e cálculo FP16. O model card informa cerca de 3,09 bilhões de parâmetros. A licença indicada é `qwen-research`; consultar seus termos antes de distribuir pesos ou mudar a finalidade do projeto.

[Model card](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) · [Quantização bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes).

**Hipótese a verificar:** a configuração consegue gerar respostas curtas com memória disponível. Inferência funcionar não prova que o futuro treino QLoRA caberá na GPU.

## 2. Conferir o kernel antes de carregar pesos

Selecionar o kernel do ambiente `dsadlp1` no VS Code. Um Python diferente pode não encontrar CUDA ou as bibliotecas. A célula seguinte mostra o executável e as versões efetivamente instaladas; não instala pacotes automaticamente.

O primeiro acesso ao Hugging Face pode baixar o tokenizador e vários GB de pesos. Encerrar outros notebooks que mantenham modelos na GPU antes do carregamento. Executar uma célula por vez nesta primeira leitura.

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python do kernel:", sys.executable)
pacotes = ["torch", "transformers", "accelerate", "bitsandbytes", "langchain-core"]
versoes = {}
for pacote in pacotes:
    try:
        versoes[pacote] = version(pacote)
        print(pacote, "->", versoes[pacote])
    except PackageNotFoundError as erro:
        raise RuntimeError(f"Falta {pacote} neste kernel. Conferir o ambiente dsadlp1.") from erro

### 2.1. Imports e dispositivo

PyTorch executa tensores e a LLM. Transformers fornece tokenizador e geração. Bitsandbytes será acionado pela configuração de quantização. LangChain entrará somente após a primeira chamada direta.

Não se suprimem warnings globalmente. Avisos de compatibilidade precisam permanecer visíveis.

In [ ]:
from pathlib import Path
import json
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Esta etapa requer PyTorch com CUDA no kernel selecionado.")

print("GPU:", torch.cuda.get_device_name(0))
memoria_livre, memoria_total = torch.cuda.mem_get_info(0)
print(f"VRAM total: {memoria_total / 1024**3:.2f} GiB")
print(f"VRAM livre agora: {memoria_livre / 1024**3:.2f} GiB")

## 3. Configurações explícitas

Um único identificador de LLM será usado. O orçamento inicial é conservador: uma pergunta, até 1.024 tokens de entrada e até 256 novos tokens. Esses limites são decisões do experimento, não o limite arquitetural do Qwen.

`RANDOM_STATE = 42` reduz variações; não assegura igualdade entre hardwares ou versões. `do_sample=False`, usado adiante, desativa amostragem. As respostas iniciais não são evidências sobre o corpus, pois ainda não existe RAG.

In [ ]:
RANDOM_STATE = 42
NOME_MODELO = "Qwen/Qwen2.5-3B-Instruct"
REVISAO_MODELO = "main"  # Na primeira execução; o commit resolvido será registrado.
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 256
set_seed(RANDOM_STATE)

# Aceita iniciar o notebook na própria pasta ou na raiz do repositório.
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

## 4. Do texto aos tokens

O tokenizador transforma texto em IDs do vocabulário. Um token pode representar parte de uma palavra, pontuação ou outros fragmentos. Não se deve converter um limite de tokens em palavras usando uma equivalência fixa.

Primeiro será examinada uma frase curta, sem carregar a LLM. O tokenizador e os pesos deverão corresponder ao mesmo checkpoint.

In [ ]:
tokenizador = AutoTokenizer.from_pretrained(NOME_MODELO, revision=REVISAO_MODELO)
if tokenizador.chat_template is None:
    raise ValueError("O checkpoint precisa fornecer o template de conversa.")

frase = "Transformers podem representar sequências de comportamento financeiro."
ids_frase = tokenizador.encode(frase, add_special_tokens=False)
tokens_frase = tokenizador.convert_ids_to_tokens(ids_frase)

print("Texto:", frase)
print("Quantidade de tokens:", len(ids_frase))
for token, identificador in zip(tokens_frase, ids_frase):
    print(repr(token), "->", identificador)
print("Texto reconstruído:", tokenizador.decode(ids_frase))

**Interpretação:** símbolos exibidos nos tokens podem codificar espaços ou bytes; o teste de leitura é a decodificação. Alterar a frase e observar a nova segmentação ajuda a compreender por que caracteres e tokens são medidas distintas.

**Checkpoint:** o tokenizador não responde à pergunta e não produz o embedding documental que será indexado no Chroma. Sua função aqui é preparar a entrada da LLM.

## 5. Construir uma conversa antes de gerar

Separaram-se duas mensagens: `system` define o comportamento pretendido e `user` contém a pergunta. Esse papel de especialista é uma instrução; não é treinamento e não certifica conhecimento.

Na primeira pergunta, solicita-se uma explicação curta sobre atenção. A aplicação ao crédito será apresentada como hipótese de experimento, sem afirmar vantagem sobre LightGBM.

In [ ]:
INSTRUCAO_SISTEMA = (
    "Responda em português, de forma didática e impessoal. "
    "Explique Transformers no contexto de estudo de risco de crédito Pessoa Física. "
    "Diferencie fatos, hipóteses e exemplos sintéticos. "
    "Não invente artigos, métricas ou resultados. Quando não souber, explicite a limitação."
)
PERGUNTA = (
    "Explique brevemente por que a atenção pode ajudar a representar 12 meses "
    "de comportamento de um cliente PF. Dê uma hipótese de aplicação e "
    "um cuidado para evitar vazamento de informação."
)
mensagens = [
    {"role": "system", "content": INSTRUCAO_SISTEMA},
    {"role": "user", "content": PERGUNTA},
]
texto_prompt = tokenizador.apply_chat_template(
    mensagens, tokenize=False, add_generation_prompt=True,
)
print(texto_prompt)

### 5.1. Inspecionar o batch

O template já contém os delimitadores da conversa; por isso a tokenização abaixo usa `add_special_tokens=False`. `input_ids` e `attention_mask` possuem formato `(B, T)`: B é o número de exemplos e T o número de tokens. Nesta chamada B = 1.

A máscara indica posições válidas, especialmente quando houver padding. Não confundir essa máscara com os pesos de atenção calculados dentro do Transformer.

In [ ]:
entradas_cpu = tokenizador(
    texto_prompt,
    return_tensors="pt",
    add_special_tokens=False,
)
quantidade_tokens_entrada = entradas_cpu["input_ids"].shape[1]
print("input_ids:", entradas_cpu["input_ids"].shape)
print("attention_mask:", entradas_cpu["attention_mask"].shape)
print("Primeiros IDs:", entradas_cpu["input_ids"][0, :20])
print("Tokens de entrada:", quantidade_tokens_entrada)
if quantidade_tokens_entrada > MAX_INPUT_TOKENS:
    raise ValueError("Prompt excede o orçamento inicial; reduza a pergunta antes de gerar.")

## 6. Entender a quantização antes de carregar

Em uma aproximação apenas para pesos, 3,09 bilhões de parâmetros em 16 bits representam cerca de 6,18 GB; em 4 bits seriam 1,545 GB. Essa conta **não é o consumo real de VRAM**: há escalas, tensores não quantizados, ativações, cache de geração e memória do runtime.

Optou-se por NF4, quantização dupla e cálculo FP16, em continuidade com o Projeto 7. Armazenar pesos em 4 bits não significa que todas as operações sejam executadas em 4 bits. Esta etapa é inferência quantizada; só haverá QLoRA quando adaptadores treináveis forem adicionados e otimizados.

O carregamento é fixado na GPU 0 para tornar a execução explícita. Se faltar memória, o erro deverá ser resolvido antes de seguir, sem trocar silenciosamente de modelo ou descarregar parte dele para CPU.

In [ ]:
configuracao_quantizacao = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

inicio_carregamento = time.perf_counter()
modelo = AutoModelForCausalLM.from_pretrained(
    NOME_MODELO,
    revision=REVISAO_MODELO,
    quantization_config=configuracao_quantizacao,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
modelo.eval()
tempo_carregamento = time.perf_counter() - inicio_carregamento
print(f"Carregamento: {tempo_carregamento:.1f} s")
print(f"Memória reportada para o modelo: {modelo.get_memory_footprint() / 1024**3:.2f} GiB")
print("Revisão resolvida:", getattr(modelo.config, "_commit_hash", None))

**Interpretação:** `eval()` coloca o modelo em modo de avaliação; não desativa sozinho o cálculo de gradientes. Na geração será utilizado também `torch.inference_mode()`.

O tempo de carregamento pode incluir download e não deve ser comparado à latência da resposta. A memória reportada pelo modelo também não é a memória total usada na placa. Não repetir esta célula com outro modelo ainda residente; reiniciar o kernel se necessário.

## 7. Gerar a primeira resposta sem RAG

Agora a pergunta tokenizada será enviada ao Qwen. A geração autorregressiva acrescenta um token por passo até encontrar uma condição de término ou alcançar o limite.

`do_sample=False` e `num_beams=1` definem geração gulosa. Isso facilita comparar chamadas, mas não garante correção. O cache de geração evita recalcular parte do trabalho anterior e ocupa memória adicional.

In [ ]:
entradas_gpu = entradas_cpu.to("cuda:0")
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
inicio_geracao = time.perf_counter()

with torch.inference_mode():
    ids_saida = modelo.generate(
        **entradas_gpu,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        pad_token_id=tokenizador.pad_token_id or tokenizador.eos_token_id,
    )

torch.cuda.synchronize()
tempo_geracao = time.perf_counter() - inicio_geracao
# A saída causal inclui o prompt; recorta-se apenas a continuação.
ids_resposta = ids_saida[0, quantidade_tokens_entrada:]
resposta = tokenizador.decode(ids_resposta, skip_special_tokens=True)
print(resposta)

### 7.1. Interpretar custo e conteúdo

A contagem abaixo mede tokens gerados, não palavras. A primeira chamada pode pagar custos de aquecimento; um único tempo não constitui benchmark. O pico de memória alocada pelo PyTorch não inclui toda a memória de outros processos.

Conferir manualmente: a resposta explica atenção? Identifica uma hipótese PF? Menciona disponibilidade temporal das variáveis? Evita afirmar ganhos inexistentes? A avaliação humana ainda será necessária mesmo quando o RAG estiver implementado.

In [ ]:
quantidade_tokens_saida = len(ids_resposta)
print("Tokens gerados:", quantidade_tokens_saida)
print(f"Tempo da geração: {tempo_geracao:.2f} s")
print(f"Tokens/s nesta chamada: {quantidade_tokens_saida / tempo_geracao:.2f}")
print(f"Pico alocado pelo PyTorch: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
if quantidade_tokens_saida == MAX_NEW_TOKENS:
    print("A resposta pode ter atingido o limite. Conferir se a explicação ficou incompleta.")

## 8. Introduzir LangChain sem esconder a geração

Até aqui, mensagens, template, tokenização e geração ficaram explícitos. LangChain será utilizado agora para organizar a instrução e a pergunta em um template reutilizável.

Essa célula não chama outra LLM, não executa RAG e não faz uma segunda geração. Sua saída será comparada às mensagens já utilizadas. A chamada ao Qwen continua visível na seção anterior; a composição completa entrará quando houver documentos para recuperar.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template_langchain = ChatPromptTemplate.from_messages([
    ("system", INSTRUCAO_SISTEMA),
    ("human", "{pergunta}"),
])
prompt_langchain = template_langchain.invoke({"pergunta": PERGUNTA})
mensagens_langchain = prompt_langchain.to_messages()

# Conversão explícita entre os nomes dos papéis do LangChain e do Qwen.
mensagens_qwen = [
    {"role": "system", "content": mensagens_langchain[0].content},
    {"role": "user", "content": mensagens_langchain[1].content},
]
texto_via_langchain = tokenizador.apply_chat_template(
    mensagens_qwen, tokenize=False, add_generation_prompt=True,
)
print(texto_via_langchain)
assert texto_via_langchain == texto_prompt
print("O template organizado pelo LangChain preservou o prompt da chamada direta.")

## 9. Registrar a execução realizada

Salvar apenas informações observadas: pergunta, resposta, configuração, versões e custo. Não registrar uma nota automática de qualidade nem declarar sucesso no domínio com base em uma demonstração.

O JSON abaixo é sobrescrito ao repetir esta célula, mantendo o exemplo simples. Para reproduzir os pesos depois, substituir `REVISAO_MODELO` pelo commit registrado e utilizar a mesma revisão para o tokenizador. Os arquivos não contêm dados reais de clientes.

In [ ]:
registro = {
    "modelo": NOME_MODELO,
    "revisao": getattr(modelo.config, "_commit_hash", None),
    "seed": RANDOM_STATE,
    "versoes": versoes,
    "gpu": torch.cuda.get_device_name(0),
    "quantizacao": "NF4, double quant, calculo FP16",
    "max_new_tokens": MAX_NEW_TOKENS,
    "prompt": texto_prompt,
    "resposta": resposta,
    "tokens_entrada": quantidade_tokens_entrada,
    "tokens_saida": quantidade_tokens_saida,
    "tempo_geracao_s": tempo_geracao,
    "pico_pytorch_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "usa_rag": False,
    "usa_adaptador": False,
}
caminho_resultado = DIRETORIO_RESULTADOS / "primeira_resposta_qwen.json"
caminho_resultado.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print("Registro salvo:", caminho_resultado)

## 10. Checkpoint e próxima etapa

| Pergunta | Resposta esperada |
|---|---|
| A instrução de especialista treinou a LLM? | Não; ela orienta a geração por contexto. |
| O tokenizador é o banco vetorial? | Não; converte texto em IDs para o modelo. |
| Quantização em 4 bits já é QLoRA? | Não; faltam adaptadores e treinamento. |
| LangChain substituiu o Qwen? | Não; organizou o prompt. |
| A resposta possui fontes verificáveis? | Não necessariamente; esta chamada não recuperou documentos. |

**Exercício:** alterar apenas a pergunta e executar novamente a partir da seção 5, pulando a seção 6 para reutilizar os pesos já carregados. Comparar tamanho do prompt, duração e qualidade da resposta. Não interpretar diferenças de uma única chamada como benchmark.

**Próximo incremento:** incorporar documentos públicos/autorizados e mostrar um PDF, seu texto e seus metadados antes de criar chunks. Depois serão estudados embeddings, cosseno, HNSW e MMR no mesmo notebook. O chat do Projeto 8 será adaptado após o fluxo de recuperação estar compreendido.

O futuro fine-tuning usará esta mesma base 3B; comparar base e adaptador, com e sem RAG, significa avaliar quatro configurações da mesma LLM, não selecionar quatro modelos. Se o hardware inviabilizar o ajuste, a limitação será registrada antes de qualquer mudança de escopo.

Ao encerrar o estudo, desligar o kernel para liberar a GPU antes de abrir a interface web. Esta primeira parte não declara os resultados do projeto completo.

# Parte II — Documentos reais e preparação do corpus
## 11. O que existe nos PDFs?

O diretório `documentos` contém artigos obtidos das publicações originais. `catalogo.json` registra URLs e hashes. O corpus inicial combina fundamentos de atenção, inadimplência de consumidores e representação de eventos bancários. PRAGMA é uma referência bancária, não prova que cada experimento do artigo seja sobre PF brasileiro. As propostas para PF deverão explicitar essa diferença.

Os artigos são usados para consulta local. A disponibilidade pública não concede automaticamente direito de redistribuir ou usar textos em treino. O dataset de ajuste abaixo contém exemplos didáticos próprios; não copia os PDFs.

**Objetivo:** abrir cada página e preservar fonte e número da página PDF. Não se pressupõe que extrair texto conserve tabelas, fórmulas ou ordem das colunas.

In [ ]:
from pypdf import PdfReader
import hashlib
catalogo = json.loads((DIRETORIO_PROJETO / "documentos/catalogo.json").read_text(encoding="utf-8"))
paginas = []
for fonte in catalogo:
    arquivo = DIRETORIO_PROJETO / "documentos" / fonte["file"]
    assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == fonte["sha256"]
    leitor = PdfReader(arquivo)
    print(fonte["title"], "->", len(leitor.pages), "páginas")
    for numero, pagina in enumerate(leitor.pages, start=1):
        texto = pagina.extract_text() or ""
        if texto.strip():
            paginas.append({"texto": texto, "source_id": fonte["id"], "title": fonte["title"], "page": numero, "url": fonte["url"]})
print("Páginas com texto:", len(paginas))
print(paginas[0]["texto"][:1200])

## 12. Chunks limitados pelo tokenizer do encoder

O encoder multilíngue transforma trechos e perguntas em vetores para busca. Ele é distinto da LLM geradora. Para evitar truncamento silencioso, os trechos serão janelas de até 96 tokens do encoder com 16 tokens de sobreposição. A divisão por tokens é simples e inspecionável, mas pode separar frases; será necessário revisar trechos recuperados.

A sobreposição introduz redundância; MMR será avaliado adiante. Páginas vazias são omitidas, não interpretadas como prova de documento sem conteúdo.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
NOME_ENCODER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
encoder = SentenceTransformer(NOME_ENCODER, device="cpu")
TAMANHO_CHUNK = min(96, encoder.max_seq_length - 2)
SOBREPOSICAO = 16
chunks = []
for pagina in paginas:
    tokens = encoder.tokenizer.encode(pagina["texto"], add_special_tokens=False)
    for inicio in range(0, len(tokens), TAMANHO_CHUNK - SOBREPOSICAO):
        texto = encoder.tokenizer.decode(tokens[inicio:inicio + TAMANHO_CHUNK], skip_special_tokens=True)
        if not texto.strip():
            continue
        registro = dict(pagina)
        registro["texto"] = texto
        registro["id"] = hashlib.sha256(f"{pagina['source_id']}:{pagina['page']}:{inicio}:{texto}".encode()).hexdigest()
        chunks.append(registro)
print("Chunks:", len(chunks))
print(chunks[5])
vetores = encoder.encode([c["texto"] for c in chunks], normalize_embeddings=True, batch_size=16, show_progress_bar=True)
print("Matriz:", vetores.shape)
print("Normas:", np.linalg.norm(vetores[:5], axis=1))

# Parte III — Banco vetorial real
## 13. Criar o Chroma persistente com HNSW e cosseno

`space="cosine"` define distância 1 − similaridade. HNSW mantém conexões entre vetores; não armazena todas as distâncias possíveis. `ef_search` controla esforço da busca, enquanto `n_results` controla a quantidade retornada.

Os IDs estáveis permitem `upsert` sem duplicar. A remoção abaixo limita-se a IDs obsoletos desta coleção do projeto, após a preparação completa do corpus. Não reutilizar a coleção dos exercícios antigos.

In [ ]:
import chromadb
cliente = chromadb.PersistentClient(path=str(DIRETORIO_PROJETO / "banco_chroma"))
colecao = cliente.get_or_create_collection(
    "transformers_credito_v1",
    configuration={"hnsw": {"space": "cosine", "max_neighbors": 16, "ef_construction": 100, "ef_search": 100}},
    metadata={"encoder": NOME_ENCODER},
)
assert colecao.metadata["encoder"] == NOME_ENCODER
ids_atuais = [c["id"] for c in chunks]
for inicio in range(0, len(chunks), 100):
    lote = chunks[inicio:inicio+100]
    colecao.upsert(
        ids=[c["id"] for c in lote], documents=[c["texto"] for c in lote],
        embeddings=vetores[inicio:inicio+100].tolist(),
        metadatas=[{k:v for k,v in c.items() if k not in ["id", "texto"]} for c in lote],
    )
obsoletos = sorted(set(colecao.get()["ids"]) - set(ids_atuais))
if obsoletos:
    colecao.delete(ids=obsoletos)
assert colecao.count() == len(chunks)
print("Registros persistidos:", colecao.count())
print("Configuração:", colecao.configuration)

## 14. Busca exata versus HNSW e seleção MMR

Com vetores normalizados, produto escalar = cosseno. A busca exata abaixo fornece uma referência para a aproximação HNSW. Recall ANN compara IDs, não avalia se os documentos respondem corretamente. Uma única pergunta é demonstração, não benchmark.

Depois da recuperação, MMR combina relevância para a pergunta e penalização de redundância. `peso=0.7` prioriza relevância. O helper compartilhado com a interface tem seu código exibido para inspeção.

In [ ]:
import sys
import inspect
if str(DIRETORIO_PROJETO) not in sys.path:
    sys.path.insert(0, str(DIRETORIO_PROJETO))
from rag import selecionar_mmr, recuperar, montar_mensagens
print(inspect.getsource(selecionar_mmr))

pergunta_rag = "Como a atenção pode representar sequências e quais cuidados existem na previsão de inadimplência de consumidores?"
vetor_consulta = encoder.encode(pergunta_rag, normalize_embeddings=True)
similaridades = vetores @ vetor_consulta
indices_exatos = np.argsort(-similaridades)[:4]
resultado_ann = colecao.query(query_embeddings=[vetor_consulta.tolist()], n_results=4)
ids_exatos = {chunks[i]["id"] for i in indices_exatos}
recall_ann = len(ids_exatos.intersection(resultado_ann["ids"][0])) / 4
print("Recall ANN@4 desta consulta:", recall_ann)
trechos = recuperar(colecao, encoder, pergunta_rag, usar_mmr=True, k=4, fetch_k=12)
for trecho in trechos:
    print(trecho["source_id"], "página", trecho["page"], trecho["texto"][:250])

# Parte IV — RAG e interface web
## 15. Fornecer evidências ao mesmo Qwen

Agora os documentos fazem parte do prompt. O modelo continua com os mesmos pesos. A instrução exige referências e separação entre resultados documentados e hipóteses; isso não garante obediência. Conferir as citações na fonte.

Para preservar legibilidade, as mensagens são mostradas antes de gerar. Se o prompt ultrapassar o orçamento, reduzir trechos conscientemente; não cortar evidências silenciosamente.

In [ ]:
from rag import gerar
mensagens_rag = montar_mensagens(pergunta_rag, trechos)
print(mensagens_rag[-1]["content"])
resposta_rag = gerar(modelo, tokenizador, mensagens_rag)
print(resposta_rag)

## 16. Abrir a interface criada

O arquivo `app.py` implementa chat Streamlit com mensagens, histórico recente, fontes expansíveis, MMR e seleção entre base e adaptador da mesma LLM. O botão de exportação preserva a conversa em JSON; o histórico não é persistido automaticamente entre sessões.

1. Concluir a indexação acima.
2. Encerrar o kernel para liberar a GPU.
3. No terminal, ativar `dsadlp1`, entrar nesta pasta e executar:

```powershell
python -m streamlit run app.py --server.address 127.0.0.1
```

Abrir a URL indicada no terminal. A opção “Ajustado” só funciona depois de salvar o adaptador na próxima parte. Reiniciar o servidor para trocar pesos sem carregar duas cópias. A interface já existe; não se trata mais de uma proposta futura.

# Parte V — Fine-tuning QLoRA, passo a passo
## 17. Dataset supervisionado próprio

O arquivo `dataset_sft.jsonl` contém um piloto pequeno de perguntas e respostas didáticas, separado por famílias de assuntos. Seu propósito é verificar tokenização, máscaras, gradientes e salvamento. **Não é suficiente para comprovar especialização**. Revisar e ampliar antes do treino definitivo. Os artigos não são copiados para o treinamento.

Será treinada apenas a resposta: tokens da instrução/prompt terão label -100. O modelo causal desloca os labels internamente para prever o próximo token. Não deslocar duas vezes. Exemplos maiores que 512 tokens são rejeitados explicitamente, sem remover a resposta por truncamento.

In [ ]:
exemplos = [json.loads(linha) for linha in (DIRETORIO_PROJETO / "dataset_sft.jsonl").read_text(encoding="utf-8").splitlines() if linha.strip()]
treino = [e for e in exemplos if e["split"] == "treino"]
validacao = [e for e in exemplos if e["split"] == "validacao"]
teste = [e for e in exemplos if e["split"] == "teste"]
print("Treino/validação/teste:", len(treino), len(validacao), len(teste))
assert set(e["familia"] for e in treino).isdisjoint(e["familia"] for e in validacao + teste)
print(treino[0])

lotes = {"treino": [], "validacao": []}
for nome_split, registros in [("treino", treino), ("validacao", validacao)]:
    for exemplo in registros:
        mensagens = [{"role": "system", "content": INSTRUCAO_SISTEMA}, {"role": "user", "content": exemplo["pergunta"]}]
        prefixo = tokenizador.apply_chat_template(mensagens, tokenize=True, add_generation_prompt=True)
        completo = tokenizador.apply_chat_template(mensagens + [{"role": "assistant", "content": exemplo["resposta"]}], tokenize=True)
        assert completo[:len(prefixo)] == prefixo
        if len(completo) > 512:
            raise ValueError(f"Exemplo longo: {exemplo['familia']}; revisar antes do treino.")
        labels = [-100] * len(prefixo) + completo[len(prefixo):]
        assert any(x != -100 for x in labels)
        lotes[nome_split].append({"input_ids": torch.tensor([completo]), "attention_mask": torch.ones(1, len(completo), dtype=torch.long), "labels": torch.tensor([labels])})
print("Shape:", lotes["treino"][0]["input_ids"].shape)
print("Labels:", lotes["treino"][0]["labels"])
print("Texto supervisionado:", tokenizador.decode([x for x in lotes["treino"][0]["labels"][0].tolist() if x != -100]))

## 18. Congelar a base e adicionar LoRA

Para W de dimensão saída × entrada, LoRA aprende uma atualização BA de rank pequeno. A base em 4 bits fica congelada. Neste piloto, adaptam-se `q_proj` e `v_proj` com rank 8 e alpha 16; não se afirma que essa seja a melhor configuração.

O bloco usa uma flag explícita porque altera o modelo em memória. Para repetir o experimento, reiniciar o kernel e carregar uma base limpa; não inserir adaptadores repetidamente. Fechar a interface antes do treino.

In [ ]:
EXECUTAR_AJUSTE = False  # Alterar para True após revisar os exemplos e encerrar o chat.
if EXECUTAR_AJUSTE:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if hasattr(modelo, "peft_config"):
        raise RuntimeError("Recarregue a base limpa antes de repetir o ajuste.")
    modelo.config.use_cache = False
    modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=True)
    config_lora = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM",
        base_model_name_or_path=NOME_MODELO,
        revision=getattr(modelo.config, "_commit_hash", None),
    )
    modelo = get_peft_model(modelo, config_lora)
    modelo.print_trainable_parameters()
else:
    print("Ajuste desativado. Nenhum peso foi treinado.")

## 19. Loop explícito de treino e validação

Batch por dispositivo = 1; acumulação = 4. Cada loss é dividida pelo número real de microbatches do grupo, incluindo o último grupo incompleto. `backward()` acumula gradientes; o otimizador atualiza só no fim do grupo. Gradient clipping limita sua norma.

Optou-se por três épocas apenas para o piloto e learning rate 0,0001. A loss de validação é ponderada pelos tokens supervisionados. O melhor adaptador é salvo pela validação, sem selecionar pelo teste. Um piloto tão pequeno não sustenta conclusões estatísticas.

Os adaptadores preparados pelo PEFT usam parâmetros treináveis em FP32; a base calcula em FP16. Não se adiciona um segundo autocast que oculte o regime de precisão.

In [ ]:
if EXECUTAR_AJUSTE:
    import random
    set_seed(RANDOM_STATE)
    otimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=1e-4)
    EPOCAS = 3
    ACUMULACAO = 4
    melhor_loss = float("inf")
    historico_treino = []
    for epoca in range(EPOCAS):
        modelo.train()
        ordem = list(range(len(lotes["treino"])))
        random.Random(RANDOM_STATE + epoca).shuffle(ordem)
        perdas = []
        for inicio in range(0, len(ordem), ACUMULACAO):
            grupo = ordem[inicio:inicio + ACUMULACAO]
            otimizador.zero_grad(set_to_none=True)
            for indice in grupo:
                batch = {k:v.to("cuda:0") for k,v in lotes["treino"][indice].items()}
                saida = modelo(**batch)
                if not torch.isfinite(saida.loss):
                    raise RuntimeError("Loss não finita. Interromper e investigar precisão/dados.")
                perdas.append(saida.loss.item())
                (saida.loss / len(grupo)).backward()
            torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.0)
            otimizador.step()
        modelo.eval()
        soma_loss, total_tokens = 0.0, 0
        with torch.no_grad():
            for lote in lotes["validacao"]:
                batch = {k:v.to("cuda:0") for k,v in lote.items()}
                quantidade = int((batch["labels"][:, 1:] != -100).sum())
                soma_loss += modelo(**batch).loss.item() * quantidade
                total_tokens += quantidade
        loss_val = soma_loss / total_tokens
        registro_epoca = {"epoca": epoca+1, "loss_treino": sum(perdas)/len(perdas), "loss_validacao": loss_val}
        historico_treino.append(registro_epoca)
        print(registro_epoca)
        if loss_val < melhor_loss:
            melhor_loss = loss_val
            modelo.save_pretrained(DIRETORIO_PROJETO / "adaptador_pf")
            tokenizador.save_pretrained(DIRETORIO_PROJETO / "adaptador_pf")
    (DIRETORIO_RESULTADOS / "historico_qlora.json").write_text(json.dumps(historico_treino, indent=2), encoding="utf-8")
    print("Melhor adaptador salvo. Reiniciar o kernel/servidor para carregá-lo na inferência.")

# Parte VI — Avaliar e entregar
## 20. Mesmo modelo, com e sem ajuste/RAG

Executar a célula abaixo em kernel limpo após carregar **ou** base **ou** adaptador pelo helper `carregar_llm`. Não manter as duas cópias na GPU. O código verifica o modo informado contra a presença de adaptadores. A avaliação usa casos reservados, nunca usados para selecionar épocas.

O arquivo preserva respostas para revisão humana. Avaliar correção, evidência, citações e utilidade de propostas; não confundir menor loss com melhoria no domínio. O piloto não autoriza divulgar o modelo como validado.

In [ ]:
EXECUTAR_AVALIACAO = False
MODO_AVALIACAO = "base"  # "ajustado" somente após recarregar o melhor adaptador salvo.
if EXECUTAR_AVALIACAO:
    assert (MODO_AVALIACAO == "ajustado") == hasattr(modelo, "peft_config")
    avaliacoes = []
    for exemplo in teste:
        for usar_rag in [False, True]:
            evidencias = recuperar(colecao, encoder, exemplo["pergunta"]) if usar_rag else []
            mensagens = montar_mensagens(exemplo["pergunta"], evidencias) if usar_rag else [
                {"role": "system", "content": INSTRUCAO_SISTEMA},
                {"role": "user", "content": exemplo["pergunta"]},
            ]
            inicio = time.perf_counter()
            resposta_teste = gerar(modelo, tokenizador, mensagens)
            avaliacoes.append({"modo": MODO_AVALIACAO, "rag": usar_rag, "pergunta": exemplo["pergunta"], "referencia": exemplo["resposta"], "resposta": resposta_teste, "fontes": evidencias, "tempo_s": time.perf_counter()-inicio})
    (DIRETORIO_RESULTADOS / f"avaliacao_{MODO_AVALIACAO}.json").write_text(json.dumps(avaliacoes, ensure_ascii=False, indent=2), encoding="utf-8")

## 21. O que foi implementado e o que precisa ser demonstrado

O código contempla corpus PDF, extração, chunks, embeddings, Chroma/HNSW, cosseno, MMR, RAG, chat e piloto QLoRA. O código estar presente não significa que o treino ou a avaliação completa já foram executados. Os resultados reais ficam nos arquivos de saída produzidos pelas células.

Próximos refinamentos dependerão dos erros observados: revisão da extração, ampliação do corpus PF, dataset supervisionado revisado, avaliação de múltiplas consultas e testes de geração. Não automatizar a expansão do dataset com respostas do próprio chat. Não publicar PDFs ou adaptadores sem conferir as respectivas condições de uso.